In [ ]:
import math
from typing import List
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import gc
import numpy as np
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LRScheduler, _enable_get_lr_call
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

def set_seed(seed=42):
    """设置所有随机种子以保证实验可复现"""
    #random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # 如果使用多GPU
    
    # 设置CUDNN以保证确定性行为
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # 设置Python哈希种子（部分情况下影响数据加载顺序）
    import os
    os.environ['PYTHONHASHSEED'] = str(seed)

# 执行设置
set_seed(42)


def epochs_count(T_0: int, T_mult: int, num: int) -> int:
    """
    根据周期数与周期膨胀率计算在几个epoch后最后一个周期恰好截止
    
    Args:
        T_0: 第一个周期的长度
        T_mult: 周期膨胀率
        num: 周期数
    
    Returns:
        达到num个周期所需的总epoch数
    """
    if num <= 0:
        return 0
    
    if T_mult == 1:
        return T_0 * num
    else:
        # 等比数列求和: T_0 + T_0*T_mult + T_0*T_mult^2 + ... + T_0*T_mult^(num-1)
        # = T_0 * (T_mult^num - 1) / (T_mult - 1)
        return int(T_0 * (T_mult ** num - 1) / (T_mult - 1))


class CosineAnnealingWarmRestartsWithSinMaxLR(LRScheduler):
    r"""Set the learning rate of each parameter group using a cosine annealing schedule
    with warm restarts and sin-adjusted maximum learning rate.

    The :math:`\eta_{max}` is dynamically adjusted using a sin function based on the
    current cycle number and total warmup cycles. :math:`T_{cur}` is the number of 
    epochs since the last restart and :math:`T_{i}` is the number of epochs between 
    two warm restarts.

    The learning rate formula is:

    .. math::
        \eta_t = \eta_{min} + \frac{1}{2}(\eta_{max}^{'} - \eta_{min})\left(1 +
        \cos\left(\frac{T_{cur}}{T_{i}}\pi\right)\right)

    where :math:`\eta_{max}^{'}` is the adjusted maximum learning rate:

    .. math::
        \eta_{max}^{'} = \eta_{max} \cdot \sin\left(\frac{num_{com} + 1}{num_{warmup} + 1}\pi\right)

    Args:
        optimizer (Optimizer): Wrapped optimizer.
        T_0 (int): Number of iterations until the first restart.
        T_mult (int, optional): A factor by which :math:`T_{i}` increases after a restart. Default: 1.
        eta_min (float, optional): Minimum learning rate. Default: 0.
        num_warmup (int, optional): Total number of warm restart cycles. Default: 10.
        last_epoch (int, optional): The index of the last epoch. Default: -1.

    Example:
        >>> optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
        >>> scheduler = CosineAnnealingWarmRestartsWithSinMaxLR(
        ...     optimizer, T_0=10, T_mult=2, eta_min=0.001, num_warmup=10
        ... )
        >>> total_epochs = epochs_count(T_0=10, T_mult=2, num=10)
        >>> for epoch in range(total_epochs):
        ...     train(...)
        ...     validate(...)
        ...     scheduler.step()
    """

    def __init__(
        self,
        optimizer: Optimizer,
        T_0: int,
        T_mult: int = 1,
        eta_min: float = 0.0,
        num_warmup: int = 10,
        last_epoch: int = -1,
    ) -> None:
        """
        Args:
            optimizer: 优化器
            T_0: 第一个热重启周期的长度
            T_mult: 周期膨胀率
            eta_min: 最小学习率
            num_warmup: 总热重启周期数
            last_epoch: 上一个epoch的索引
        """
        if T_0 <= 0 or not isinstance(T_0, int):
            raise ValueError(f"Expected positive integer T_0, but got {T_0}")
        if T_mult < 1 or not isinstance(T_mult, int):
            raise ValueError(f"Expected integer T_mult >= 1, but got {T_mult}")
        if not isinstance(eta_min, (float, int)):
            raise ValueError(
                f"Expected float or int eta_min, but got {eta_min} of type {type(eta_min)}"
            )
        if num_warmup <= 0 or not isinstance(num_warmup, int):
            raise ValueError(f"Expected positive integer num_warmup, but got {num_warmup}")
        
        self.T_0 = T_0
        self.T_i = T_0
        self.T_mult = T_mult
        self.eta_min = eta_min
        self.T_cur = last_epoch
        self.num_warmup = num_warmup  # 总热重启周期数
        super().__init__(optimizer, last_epoch)

    def get_num_warmup(self) -> int:
        """
        计算总热重启周期数
        
        Returns:
            总热重启周期数
        """
        return self.num_warmup

    def get_num_com(self) -> int:
        """
        计算在即将开始热重启周期时，已经完成的周期数
        
        Returns:
            已完成的周期数
        """
        if self.last_epoch < self.T_0:
            return 0
        
        if self.T_mult == 1:
            return min(self.last_epoch // self.T_0, self.num_warmup)
        else:
            # 计算已完成的完整周期数
            n = 0
            total_epochs = self.T_0
            while total_epochs <= self.last_epoch and n < self.num_warmup:
                n += 1
                if n < self.num_warmup:
                    total_epochs += self.T_0 * (self.T_mult ** n)
            return min(n, self.num_warmup)

    def _get_current_max_lr(self, base_lr: float) -> float:
        """
        计算当前周期的最大学习率
        
        在热重启周期开始时，使用sin函数调整最大学习率:
            max_lr * sin((num_com + 1) / (num_warmup + 1) * π)
        
        Args:
            base_lr: 原始最大学习率
            
        Returns:
            调整后的最大学习率
        """
        num_com = self.get_num_com()
        # 使用sin函数调整最大学习率
        ratio = (num_com + 1) / (self.num_warmup + 1)
        sin_factor = math.sin(ratio * math.pi)
        return base_lr * sin_factor

    def get_lr(self) -> List[float]:
        """
        计算下一个学习率
        
        Returns:
            各参数组的学习率列表
        """
        # 计算当前周期的最大学习率（使用sin调整）
        current_max_lrs = [self._get_current_max_lr(base_lr) for base_lr in self.base_lrs]
        
        return [
            self.eta_min
            + (current_max_lr - self.eta_min)
            * (1 + math.cos(math.pi * self.T_cur / self.T_i))
            / 2
            for current_max_lr in current_max_lrs
        ]

    def step(self, epoch=None) -> None:
        """
        执行一步调度
        
        Args:
            epoch: 当前epoch，如果为None则自动递增
        """
        if epoch is None and self.last_epoch < 0:
            epoch = 0

        if epoch is None:
            epoch = self.last_epoch + 1
            self.T_cur = self.T_cur + 1
            if self.T_cur >= self.T_i:
                self.T_cur = self.T_cur % self.T_i
                self.T_i = self.T_i * self.T_mult
        else:
            if epoch < 0:
                raise ValueError(f"Expected non-negative epoch, but got {epoch}")
            if epoch >= self.T_0:
                if self.T_mult == 1:
                    self.T_cur = epoch % self.T_0
                else:
                    n = int(
                        math.log(
                            (epoch / self.T_0 * (self.T_mult - 1) + 1), self.T_mult
                        )
                    )
                    self.T_cur = epoch - self.T_0 * (self.T_mult**n - 1) / (
                        self.T_mult - 1
                    )
                    self.T_i = self.T_0 * self.T_mult ** (n)
            else:
                self.T_i = self.T_0
                self.T_cur = epoch
        self.last_epoch = math.floor(epoch)

        with _enable_get_lr_call(self):
            for param_group, lr in zip(
                self.optimizer.param_groups, self.get_lr(), strict=True
            ):
                param_group['lr'] = lr

        self._last_lr = [group['lr'] for group in self.optimizer.param_groups]

def run_training(name, total_epochs, use_custom_scheduler=True):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n>>> 启动实验: {name}| 设备: {device}")

    # 数据增强
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408),
                             (0.2675, 0.2565, 0.2761)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408),
                             (0.2675, 0.2565, 0.2761)),
    ])

    trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=False, transform=transform_train)
    testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=False, transform=transform_test)

    trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=20)
    testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=20)

    model = torchvision.models.resnet18(num_classes=100).to(device)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    # 去掉第一层池化：32x32 经不起这么多次减半
    model.maxpool = nn.Identity()
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer_1 = optim.AdamW(model.parameters(), lr=math.pi * 2e-3 / 2, weight_decay=1e-4)
    optimizer_2 = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

    # 调度器
    if use_custom_scheduler:
        optimizer = optimizer_1
        scheduler = CosineAnnealingWarmRestartsWithSinMaxLR(
            optimizer=optimizer,
            T_0=10,
            T_mult=1,
            eta_min=1e-6,
            num_warmup=100
        )
    else:
        optimizer = optimizer_2
        # 标准 SGDR：使用 optimizer 的初始 lr 作为 eta_max
        scheduler = CosineAnnealingWarmRestarts(
            optimizer=optimizer,
            T_0=10,
            T_mult=1,
            eta_min=1e-6
        )


    history = {'loss': [], 'train_acc': [], 'test_acc': [], 'lr': []}

    for epoch in range(total_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        current_lr = optimizer.param_groups[0]['lr']

        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        scheduler.step()
        train_acc = 100. * correct / total
        avg_loss = running_loss / len(trainloader)

        # 验证
        model.eval()
        test_correct, test_total = 0, 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                test_total += labels.size(0)
                test_correct += predicted.eq(labels).sum().item()

        test_acc = 100. * test_correct / test_total

        history['loss'].append(avg_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        history['lr'].append(current_lr)

        # 打印日志
        print(f"Epoch {epoch+1:03d}/{total_epochs} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}% | LR: {current_lr:.5f}")

    # --- 统一绘图 ---
    fig, axes = plt.subplots(1, 3, figsize=(18,5))
    axes[0].plot(range(total_epochs), history['lr'], color='blue'); axes[0].set_title("Learning Rate"); axes[0].set_xlabel("Epoch")
    axes[1].plot(range(total_epochs), history['loss'], color='orange'); axes[1].set_title("Train Loss"); axes[1].set_xlabel("Epoch"); axes[1].set_yscale('log')
    axes[2].plot(range(total_epochs), history['test_acc'], color='green'); axes[2].set_title("Test Accuracy"); axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("%")
    plt.savefig(f"{name}_cifar100_lrtuned_T1.png")
    plt.show()

    del model, optimizer; gc.collect(); torch.cuda.empty_cache()
    return history

def save_history_to_npz(history, name):
    filename = f"history_{name}_cifar100_lrtuned_T1.npz"
    # 将列表转换为 numpy 数组存储
    np.savez(filename, 
             loss=np.array(history['loss']),
             train_acc=np.array(history['train_acc']),
             test_acc=np.array(history['test_acc']),
             lr=np.array(history['lr']))
    print(f"历史数据已保存至: {filename}")
# --- 4. 运行实验 ---
# 计算总epoch，使得训练恰好在完整周期末尾

if __name__ == '__main__':
    EPOCHS = epochs_count(10, 1, 100)
    print(f"总 epochs: {EPOCHS}")
    
    # 实验1：你的 Sin 调整版本
    history_sin = run_training('Sin-SGDR', EPOCHS, use_custom_scheduler=True)
    save_history_to_npz(history_sin, 'Sin-SGDR')
    
    # 实验2：标准 PyTorch SGDR
    history_standard = run_training('Standard-SGDR', EPOCHS, use_custom_scheduler=False)
    save_history_to_npz(history_standard, 'Standard-SGDR')
    
    # 可选：画对比图（学习率曲线）
    fig, axes = plt.subplots(1, 3, figsize=(14,5))
    axes[0].plot(history_sin['lr'], label='Sin-adjusted max_lr', color='blue')
    axes[0].plot(history_standard['lr'], label='Standard fixed max_lr', color='red', linestyle='--')
    axes[0].set_title('LR Schedule Comparison'); axes[0].legend(); axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Learning Rate')
    
    axes[1].plot(history_sin['test_acc'], label='Sin-SGDR', color='blue')
    axes[1].plot(history_standard['test_acc'], label='Standard-SGDR', color='red', linestyle='--')
    axes[1].set_title('Test Accuracy Comparison'); axes[1].legend(); axes[1].set_xlabel('Epoch')

    axes[2].plot(history_sin['loss'], label='Sin-SGDR', color='blue')
    axes[2].plot(history_standard['loss'], label='Standard SGDR', color='red', linestyle='--')
    axes[2].set_title('Loss Curve Comparison'); axes[2].legend(); axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Loss')
    plt.savefig('cifar100_lrtuned_T1.png')
    plt.show()
